# STRATA Tutorial: Conformal Prediction on Heterogeneous Infrastructure Graphs

This notebook walks through the full STRATA pipeline step by step:

1. **Data generation** — synthetic multi-utility infrastructure graph
2. **Model training** — heterogeneous GNN with per-type output heads
3. **Conformal calibration** — Mondrian, CHMP, MetaCalibrator, Ensemble
4. **Evaluation** — coverage, width, ECE, per-type analysis
5. **Streaming** — online conformal prediction with adaptive alpha
6. **Explainability** — uncertainty attribution and calibration curves

**Requirements:** `pip install -e ".[dev]"` from the STRATA root directory.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from hetero_conformal import (
    generate_synthetic_infrastructure,
    HeteroGNN,
    HeteroConformalCalibrator,
    PropagationAwareCalibrator,
    MetaCalibrator,
    EnsembleHeteroGNN,
    EnsembleCalibrator,
    StreamingConformalCalibrator,
    AdaptiveConformalCalibrator,
    marginal_coverage,
    type_conditional_coverage,
    mean_interval_width,
    calibration_error,
    interval_decomposition,
    calibration_curve_data,
    uncertainty_attribution,
    coverage_by_feature_bin,
)
from hetero_conformal.experiment import ExperimentConfig, train_model

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print("Imports OK")

## 1. Generate a Synthetic Infrastructure Graph

STRATA models three coupled infrastructure types:
- **Power** (200 nodes, tree topology)
- **Water** (150 nodes, grid/mesh)
- **Telecom** (100 nodes, star-hub)

Cross-utility coupling edges connect nodes across types with probability 0.3.

In [ ]:
graph = generate_synthetic_infrastructure(
    n_power=200, n_water=150, n_telecom=100, seed=SEED,
)
print(graph.summary())
print(f"\nNode types: {list(graph.node_features.keys())}")
print(f"Edge types: {list(graph.edge_index.keys())}")
print(f"Feature dim: {next(iter(graph.node_features.values())).shape[1]}")

### Inspect the data splits

STRATA uses a 60/20/20 train/calibration/test split. The calibration set is used
for conformal score computation (never for training).

In [ ]:
for nt in graph.node_masks:
    masks = graph.node_masks[nt]
    n = len(masks['train'])
    print(f"{nt}: {n} nodes — "
          f"train={masks['train'].sum()}, "
          f"cal={masks['cal'].sum()}, "
          f"test={masks['test'].sum()}")

## 2. Train a Heterogeneous GNN

The `HeteroGNN` uses per-edge-type weight matrices (R-GCN style) with residual
connections. Per-type output heads produce scalar risk predictions in [0, 1].

In [ ]:
config = ExperimentConfig(
    hidden_dim=64, num_layers=3, epochs=100, patience=20,
)

in_dims = {nt: f.shape[1] for nt, f in graph.node_features.items()}
edge_types = list(graph.edge_index.keys())

model = HeteroGNN(
    in_dims=in_dims,
    hidden_dim=config.hidden_dim,
    num_layers=config.num_layers,
    edge_types=edge_types,
    dropout=config.dropout,
)

model, losses, _ = train_model(model, graph, config)
print(f"Training complete: {len(losses)} epochs, final loss = {losses[-1]:.4f}")

In [ ]:
# Plot training loss
plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("HeteroGNN Training Convergence")
plt.tight_layout()
plt.show()

### Get predictions

In [ ]:
model.eval()
with torch.no_grad():
    x = {nt: torch.tensor(f, dtype=torch.float32)
         for nt, f in graph.node_features.items()}
    ei = {et: torch.tensor(e, dtype=torch.long)
          for et, e in graph.edge_index.items()}
    preds = {nt: p.numpy() for nt, p in model(x, ei, graph.num_nodes).items()}

train_masks = {nt: m["train"] for nt, m in graph.node_masks.items()}
cal_masks = {nt: m["cal"] for nt, m in graph.node_masks.items()}
test_masks = {nt: m["test"] for nt, m in graph.node_masks.items()}

for nt in preds:
    print(f"{nt}: pred range [{preds[nt].min():.3f}, {preds[nt].max():.3f}]")

## 3. Conformal Calibration

We compare four calibration strategies:

| Method | Normalization $\sigma_i$ |
|--------|------------------------|
| **Vanilla (Mondrian)** | $\sigma_i = 1$ (uniform) |
| **CHMP (Propagation)** | $\sigma_i = 1 + \lambda \bar{r}_{\mathcal{N}(i)}$ |
| **MetaCalibrator** | $\sigma_i = \text{MLP}([x_i, \hat{y}_i, \text{neighbor stats}])$ |
| **Ensemble** | $\sigma_i = 1 + \lambda \sqrt{\text{Var}(\hat{y}_i)}$ |

In [ ]:
alpha = 0.10  # Target: 90% coverage
results = {}

# --- Vanilla (Mondrian split conformal) ---
vanilla = HeteroConformalCalibrator(alpha=alpha)
vanilla.calibrate(preds, graph.node_labels, cal_masks)
results["Vanilla"] = vanilla.predict(preds, test_masks)
print("Vanilla calibrated")

# --- CHMP (Propagation-Aware) ---
prop = PropagationAwareCalibrator(alpha=alpha, neighborhood_weight=0.3)
prop.calibrate_with_propagation(
    preds, graph.node_labels, cal_masks, train_masks,
    graph.edge_index, graph.num_nodes,
)
results["CHMP"] = prop.predict(preds, test_masks)
print("CHMP calibrated")

# --- MetaCalibrator ---
meta = MetaCalibrator(alpha=alpha, meta_epochs=100)
meta.calibrate(
    graph.node_features, preds, graph.node_labels,
    cal_masks, train_masks, graph.edge_index, graph.num_nodes,
)
results["Meta"] = meta.predict(preds, test_masks)
print("MetaCalibrator calibrated")

## 4. Evaluate Coverage, Width, and ECE

In [ ]:
print(f"{'Method':<16} {'Coverage':>10} {'Width':>10} {'ECE':>10}")
print("-" * 50)

for name, result in results.items():
    cov = marginal_coverage(result, graph.node_labels, test_masks)
    width = mean_interval_width(result)
    ece = calibration_error(result, graph.node_labels, test_masks)
    print(f"{name:<16} {cov:>10.3f} {width:>10.3f} {ece:>10.4f}")

In [ ]:
# Per-type coverage
print("Per-type coverage:")
for name, result in results.items():
    tc = type_conditional_coverage(result, graph.node_labels, test_masks)
    parts = ", ".join(f"{nt}: {c:.3f}" for nt, c in tc.items())
    print(f"  {name:<16} {parts}")

### Visualize interval widths by node type

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 4), sharey=True)
if len(results) == 1:
    axes = [axes]

for ax, (name, result) in zip(axes, results.items()):
    for nt in result:
        widths = result[nt]['upper'] - result[nt]['lower']
        ax.hist(widths, bins=30, alpha=0.6, label=nt)
    ax.set_title(name)
    ax.set_xlabel("Interval Width")
    ax.legend()

axes[0].set_ylabel("Count")
plt.suptitle("Prediction Interval Width Distributions")
plt.tight_layout()
plt.show()

## 5. Streaming Conformal Prediction

STRATA supports online conformal prediction via two approaches:
- **StreamingConformalCalibrator**: sliding window of recent scores
- **AdaptiveConformalCalibrator**: Gibbs & Candès (2021) ACI with adaptive alpha

In [ ]:
stream = StreamingConformalCalibrator(alpha=0.10, window_size=200)

# Feed calibration data as a stream
for nt in cal_masks:
    mask = cal_masks[nt]
    stream.update(
        {nt: preds[nt][mask]},
        {nt: graph.node_labels[nt][mask]},
    )

# Predict on test set
stream_result = stream.predict({nt: preds[nt][test_masks[nt]] for nt in test_masks})
stream_cov = marginal_coverage(
    stream_result,
    {nt: graph.node_labels[nt][test_masks[nt]] for nt in test_masks},
)
print(f"Streaming coverage: {stream_cov:.3f}")
print(f"Window sizes: {stream.window_sizes}")

In [ ]:
# Adaptive conformal inference
aci = AdaptiveConformalCalibrator(alpha=0.10, gamma=0.01)

for nt in cal_masks:
    mask = cal_masks[nt]
    aci.update(
        {nt: preds[nt][mask]},
        {nt: graph.node_labels[nt][mask]},
    )

aci_result = aci.predict({nt: preds[nt][test_masks[nt]] for nt in test_masks})
aci_cov = marginal_coverage(
    aci_result,
    {nt: graph.node_labels[nt][test_masks[nt]] for nt in test_masks},
)
print(f"ACI coverage: {aci_cov:.3f}")
print(f"Effective alpha: {aci.current_alpha:.4f} (target: {aci.alpha_target:.2f})")

# Plot alpha history
plt.figure(figsize=(8, 3))
plt.plot(aci.alpha_history)
plt.axhline(y=alpha, color='r', linestyle='--', label=f'Target α={alpha}')
plt.xlabel("Update Step")
plt.ylabel("Effective α")
plt.title("Adaptive Conformal Inference — Alpha Trajectory")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Explainability

STRATA provides tools to understand *why* intervals are wide or narrow.

In [ ]:
# Interval decomposition: how much of the width comes from sigma vs quantile
chmp_result = results["CHMP"]
if hasattr(prop, '_sigma'):
    decomp = interval_decomposition(chmp_result, prop._sigma, test_masks)
    for nt, d in decomp.items():
        print(f"{nt}: base_quantile={d['base_quantile_contribution']:.3f}, "
              f"sigma_contribution={d['sigma_contribution']:.3f}, "
              f"total_width={d['total_width']:.3f}")

In [ ]:
# Uncertainty attribution: which features correlate with high sigma?
if hasattr(prop, '_sigma'):
    attrib = uncertainty_attribution(
        prop._sigma, graph.node_features, test_masks, top_k=3,
    )
    for nt, info in attrib.items():
        if len(info['top_features']) > 0:
            print(f"{nt}:")
            for feat, corr in zip(info['top_features'], info['top_correlations']):
                print(f"  Feature {feat}: correlation = {corr:.3f}")

In [ ]:
# Calibration curve: expected vs actual coverage across alpha levels
cal_curve = calibration_curve_data(
    results["CHMP"], graph.node_labels, test_masks, n_alpha_points=10,
 )

plt.figure(figsize=(6, 5))
for nt in cal_curve:
    expected = cal_curve[nt]['expected_coverage']
    actual = cal_curve[nt]['actual_coverage']
    plt.plot(expected, actual, 'o-', label=nt)

plt.plot([0.5, 1.0], [0.5, 1.0], 'k--', alpha=0.5, label='Perfect')
plt.xlabel("Expected Coverage (1 - α)")
plt.ylabel("Actual Coverage")
plt.title("Calibration Curve")
plt.legend()
plt.tight_layout()
plt.show()

## Summary

This tutorial demonstrated the full STRATA pipeline:

- All calibration methods maintain **valid marginal coverage** (≥ 90%)
- CHMP **redistributes** interval widths based on neighborhood difficulty
- Streaming/ACI methods enable **online** conformal prediction
- Explainability tools show **which features drive uncertainty**

For more details, see the [STRATA paper](../paper/strata-paper.md) and [API reference](../README.md).